# Programação Paralela e Distribuída
Segunda Atividade Avaliativa - 19/10/2026 - Etapa 2 **(vale 3,0 pontos)**

Nome: Eduardo Santiago Rosalin

RA:802539

## Orientações

Atividade em dupla. É vedada a troca de mensagens de qualquer natureza entre grupos. Duração: até 2 aulas.

Com base nas respostas da Etapa 1, paralelize o problema com OpenMP. Serão observados:

* Uso adequado de threads (resultado correto em todas as execuções, distribuição do trabalho, sincronização);
* Organização e clareza do código, incluindo comentários;
* Coerência entre a proposta escrita na Etapa 1 e a implementação; e
* Capacidade de explicar o funcionamento do código desenvolvido (se solicitado).

Regras:

1. **Não altere a geração dos dados** (`init()` e os `#define`). O resultado deve ser **idêntico** à saída de referência de cada problema, com 1, 2 e 3 threads e em todas as execuções.
2. Rodar várias vezes e obter o resultado certo **não prova** que o código está correto. Justifique a correção pelas cláusulas usadas.
3. Execute todas as células e **entregue o notebook com as saídas**. Preencha as tabelas de tempo com os valores que aparecem nas suas saídas.
4. Se a implementação mostrar que alguma resposta da Etapa 1 estava errada, explique na célula de análise. Isso conta a favor.
5. Caso utilize IA, cole os prompts **completos** na última seção (links não são aceitos).

Entrega: Arquivo → Fazer download → Baixar o .ipynb, com o nome `nomedosalunos.ipynb`. Apenas um da dupla envia.

## Problema 1 – Classificação de números

O programa classifica cada número de 2 a `N` pela soma `s` dos seus divisores próprios: **perfeito** (`s == n`), **abundante** (`s > n`) ou **deficiente** (`s < n`). Entre os abundantes, encontra o maior excesso `s - n` e o número que o possui (em caso de empate, o menor `n`).

Saída esperada (o tempo varia):
```
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
```

### Versão sequencial (não altere)

In [5]:
%%writefile p1_seq.c
#include <stdio.h>
#include <omp.h>

#define N 80000        // classifica todos os números de 2 a N

int main() {
    int n, d;
    long s;                    // soma dos divisores próprios de n
    long perfeitos = 0;        // s == n
    long abundantes = 0;       // s > n
    long deficientes = 0;      // s < n
    long maior_exc = 0;        // maior excesso (s - n) entre os abundantes
    int  n_maior = 0;          // número que tem o maior excesso (empate: o menor n)

    double t0 = omp_get_wtime();

    for (n = 2; n <= N; n++) {
        s = 0;
        for (d = 1; d <= n / 2; d++)
            if (n % d == 0)
                s += d;

        if (s == n)
            perfeitos++;
        else if (s > n) {
            abundantes++;
            if (s - n > maior_exc) {
                maior_exc = s - n;
                n_maior = n;
            }
        } else
            deficientes++;
    }

    double t1 = omp_get_wtime();
    printf("Perfeitos   = %ld\n", perfeitos);
    printf("Abundantes  = %ld\n", abundantes);
    printf("Deficientes = %ld\n", deficientes);
    printf("Maior excesso = %ld (n = %d)\n", maior_exc, n_maior);
    printf("Tempo = %.2f s\n", t1 - t0);
    return 0;
}

Writing p1_seq.c


In [6]:
!gcc -O3 -fopenmp p1_seq.c -o p1_seq && ./p1_seq

Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 6.48 s


### Versão paralela

Para comparar formas de distribuir o trabalho sem recompilar, use `schedule(runtime)` no laço: assim a variável de ambiente `OMP_SCHEDULE` escolhe a política na hora de executar. Sem `schedule(runtime)`, a variável `OMP_SCHEDULE` não tem efeito.

In [32]:
%%writefile p1_par.c
// VERSÃO PARALELA — parta do código sequencial abaixo.
// Comente cada diretiva e cláusula: o que faz e por que é necessária.

#include <stdio.h>
#include <omp.h>

#define N 80000        // classifica todos os números de 2 a N

int main() {
    int n, d;
    long s;

    long perfeitos = 0;
    long abundantes = 0;
    long deficientes = 0;

    long maior_exc = 0;
    int n_maior = 0;

    double t0 = omp_get_wtime();

    #pragma omp parallel private(s, d) \
        reduction(+:perfeitos, abundantes, deficientes)
    {
        long maior_exc_t = 0;
        int n_maior_t = 0;

        #pragma omp for schedule(guided)
        for (n = 2; n <= N; n++) {
            s = 0;

            for (d = 1; d <= n / 2; d++) {
                if (n % d == 0)
                    s += d;
            }

            if (s == n) {
                perfeitos++;
            }
            else if (s > n) {
                abundantes++;

                long excesso = s - n;

                if (excesso > maior_exc_t ||
                    (excesso == maior_exc_t && n < n_maior_t)) {

                    maior_exc_t = excesso;
                    n_maior_t = n;
                }
            }
            else {
                deficientes++;
            }
        }

        #pragma omp critical
        {
            if (maior_exc_t > maior_exc ||
                (maior_exc_t == maior_exc &&
                 n_maior_t != 0 &&
                 (n_maior == 0 || n_maior_t < n_maior))) {

                maior_exc = maior_exc_t;
                n_maior = n_maior_t;
            }
        }
    }

    double t1 = omp_get_wtime();

    printf("Perfeitos   = %ld\n", perfeitos);
    printf("Abundantes  = %ld\n", abundantes);
    printf("Deficientes = %ld\n", deficientes);
    printf("Maior excesso = %ld (n = %d)\n", maior_exc, n_maior);
    printf("Tempo = %.2f s\n", t1 - t0);

    return 0;
}

Overwriting p1_par.c


In [33]:
!gcc -O3 -fopenmp p1_par.c -o p1_par
!for s in static dynamic; do for t in 1 2 3; do echo "== schedule=$s, $t thread(s)"; OMP_SCHEDULE=$s OMP_NUM_THREADS=$t ./p1_par; done; done

== schedule=static, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.99 s
== schedule=static, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.32 s
== schedule=static, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.79 s
== schedule=dynamic, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.99 s
== schedule=dynamic, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.32 s
== schedule=dynamic, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.77 s


In [17]:
# Repetição: o resultado precisa ser o mesmo em todas as execuções
!for r in 1 2 3 4 5; do OMP_SCHEDULE=dynamic OMP_NUM_THREADS=2 ./p1_par | sed -n 4p; done

Maior excesso = 156320 (n = 75600)
Maior excesso = 156320 (n = 75600)
Maior excesso = 156320 (n = 75600)
Maior excesso = 156320 (n = 75600)
Maior excesso = 156320 (n = 75600)


### Tempos e análise

Preencha com os tempos das suas saídas. Se usou outra política além de `static` e `dynamic`, acrescente uma linha.

1. O resultado foi idêntico ao esperado em todas as execuções e com todas as políticas? Como vocês garantiram que `n_maior` sai correto?


3. O que mudou em relação ao planejamento da Etapa 1, se algo mudou?

*Respostas:*

-----
pragma omp parallel for schedule(runtime) private(s, d) reduction(+:perfeitos, abundantes, deficientes)

| Versão | 1 thread | 2 threads | 3 threads |
| --- | --- | --- | --- |
| Sequencial | – | – | – |
| `static` | 5.86 s | 4.84 s | 4.04 s |
| `dynamic` | 6.07 s | 3.79 s | 3.83 s |
| `guided` | – | – | – |


parallel com critical
| Versão | 1 thread | 2 threads | 3 threads |
| --- | --- | --- | --- |
| Sequencial | – | – | – |
| `static` | 6.02 s | 4.40 s | 4.11 s |
| `dynamic` | 6.07 s | 3.75 s | 3.81 s |
| `guided` | – | – | – |

omp for com ordered
| Versão | 1 thread | 2 threads | 3 threads |
| --- | --- | --- | --- |
| Sequencial | – | – | – |
| `static` | 5.65 s | 5.25 s | 4.05 s |
| `dynamic` | 6.15 s | 4.04 s | 5.34 s |
| `guided` | – | – | – |


Guided scheduling
| Versão | 1 thread | 2 threads | 3 threads |
| --- | --- | --- | --- |
| Sequencial | – | – | – |
| `static` | 4.99 s | 5.32 s | 3.79 s |
| `dynamic` | 4.99 s | 5.32 s | 3.77 s |
| `guided` | – | – | – |



- O menor tempo registrado foi 3,75 s, com dynamic, 2 threads e critical.

- Todos os testes produziram os mesmos resultados numéricos.

- A condição de corrida foi evitada usando o pragma omp critical

- E houve ganho de performance ao usar variaveis locais e diminuir o acesso a areas criticas

## Uso de IA

Se usaram IA, colem aqui os prompts **completos** e as respostas relevantes (links não são aceitos). Se não usaram, escrevam "Não utilizamos IA".


### O prompt usado:


Separe os temspode de execução nesse ofrmato: | Versão | 1 thread | 2 threads | 3 threads |
| --- | --- | --- | --- |
| Sequencial |  | – | – |
| `static` |  |  |  |
| `dynamic` |  |  |  |
| `guided` |  |  |  |

Os resultados:

pragma omp parallel for schedule(runtime) private(s, d) reduction(+:perfeitos, abundantes, deficientes)


== schedule=static, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.86 s
== schedule=static, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.84 s
== schedule=static, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.04 s
== schedule=dynamic, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 6.07 s
== schedule=dynamic, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.79 s
== schedule=dynamic, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.83 s


pragma omp parallel private(s, d) reduction(+:perfeitos, abundantes, deficientes)
com #pragma omp critical

== schedule=static, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 6.02 s
== schedule=static, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.40 s
== schedule=static, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.11 s
== schedule=dynamic, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 6.07 s
== schedule=dynamic, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.75 s
== schedule=dynamic, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.81 s


Com #pragma omp for schedule(runtime) ordered


== schedule=static, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.65 s
== schedule=static, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.25 s
== schedule=static, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.05 s
== schedule=dynamic, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 6.15 s
== schedule=dynamic, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.04 s
== schedule=dynamic, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.34 s

Com guided scheduling:

== schedule=static, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.99 s
== schedule=static, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.32 s
== schedule=static, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.79 s
== schedule=dynamic, 1 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 4.99 s
== schedule=dynamic, 2 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 5.32 s
== schedule=dynamic, 3 thread(s)
Perfeitos   = 4
Abundantes  = 19841
Deficientes = 60154
Maior excesso = 156320 (n = 75600)
Tempo = 3.77 s

Links usados:

https://610yilingliu.github.io/2020/07/15/ScheduleinOpenMP/

https://stackoverflow.com/questions/62606952/openmp-parallel-for-with-ordered-and-critical-directives

https://www.openmp.org/wp-content/uploads/SC17-Kale-LoopSchedforOMP_BoothTalk.pdf

https://medium.com/@genrunic/parallel-loop-with-openmp-scheduling-parallel-and-dsitributed-computing-ffa30f3eaf30

https://stackoverflow.com/questions/62606952/openmp-parallel-for-with-ordered-and-critical-directives

https://www.openmp.org/wp-content/uploads/openmp-webinar-vanderPas-20210318.pdf

https://stackoverflow.com/questions/10850155/whats-the-difference-between-static-and-dynamic-schedule-in-openmp